In [2]:
# =============================================================================
# CinePredict – End-to-End ML Pipeline
# =============================================================================
# Predicts three targets using only pre-release movie metadata:
#   1. Verdict label   (Hit / Average / Flop)  → XGBClassifier + LogisticRegression
#   2. Box office revenue (USD)                → XGBRegressor  + LinearRegression
#   3. IMDb-style audience rating (0–10)       → XGBRegressor  + LinearRegression
#
# Algorithms used (max 2 per course rules):
#   Algorithm 1 – XGBoost  (XGBClassifier / XGBRegressor)
#   Algorithm 2 – scikit-learn linear baselines (LogisticRegression / LinearRegression)
#
# Split strategy: temporal (by release_year) to avoid future-data leakage.
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1: Imports & constants
# ─────────────────────────────────────────────────────────────────────────────

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend – safe for scripts
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix, mean_squared_error, mean_absolute_error, r2_score,
)
from sklearn.model_selection import RandomizedSearchCV

from xgboost import XGBClassifier, XGBRegressor

# Optional SHAP – set to False to skip entirely
USE_SHAP = True
try:
    import shap
except ImportError:
    USE_SHAP = False
    print("[INFO] SHAP not installed – skipping SHAP analysis.")

warnings.filterwarnings("ignore")

# ── Adjust this path to point at your CSV ────────────────────────────────────
CSV_PATH = "cinepredict_movies_clean.csv"
OUTPUT_DIR = "cinepredict_outputs"   # folder for plots + saved models
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verdict ROI thresholds (project definition)
HIT_THRESHOLD = 2.5
AVG_THRESHOLD = 1.5

# Revenue regression: train on log-scale for numerical stability,
# then exponentiate back to USD for evaluation.
TRAIN_REVENUE_LOG = True

# Random seed for reproducibility
SEED = 42

# Temporal split quantiles
TRAIN_QUANTILE = 0.70   # oldest 70 % of years → train
VAL_QUANTILE   = 0.85   # next 15 % of years   → val
# remaining 15 % → test


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2: Load data
# ─────────────────────────────────────────────────────────────────────────────
def load_data(csv_path: str) -> pd.DataFrame:
    """Load the cleaned CinePredict CSV and do a quick sanity-check."""
    df = pd.read_csv(csv_path)
    print(f"[load_data] Loaded {len(df):,} rows × {df.shape[1]} columns.")
    print(f"            Year range: {df['release_year'].min()} – {df['release_year'].max()}")
    print(f"            Columns: {list(df.columns)}\n")
    assert df.isnull().sum().sum() == 0, "Unexpected NaNs – check your CSV!"
    return df


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3: Construct targets
# ─────────────────────────────────────────────────────────────────────────────
def add_targets(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds three target columns to df (in-place copy):
      • verdict_label   – 'Hit' / 'Average' / 'Flop'
      • revenue_usd     – already present; kept as-is
      • imdb_rating     – already present; kept as-is
      • y_revenue_log   – log1p(revenue_usd), used internally for training
    """
    df = df.copy()

    # ── 3a. Verdict label ──────────────────────────────────────────────────
    # roi = revenue / budget (simple ratio, not net profit)
    # Hit    : roi >= 2.5  (movie earned 2.5× its budget)
    # Average: 1.5 <= roi < 2.5
    # Flop   : roi < 1.5
    roi = df["revenue_usd"] / df["budget"]
    df["verdict_label"] = np.where(
        roi >= HIT_THRESHOLD, "Hit",
        np.where(roi >= AVG_THRESHOLD, "Average", "Flop")
    )
    print("[add_targets] Verdict label distribution:")
    print(df["verdict_label"].value_counts(), "\n")

    # ── 3b. Log revenue (for stable regression) ───────────────────────────
    # log1p avoids log(0); inverse is np.expm1
    df["y_revenue_log"] = np.log1p(df["revenue_usd"])

    return df


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4: Feature engineering & selection
# ─────────────────────────────────────────────────────────────────────────────
NUMERIC_FEATURES = [
    "budget", "budget_log",
    "release_year", "release_month",
    "num_genres",
    "director_avg_past_rating", "lead_actor_avg_rating",
    "sidecast_avg_rating",
    "genre_trend_score",
]

GENRE_ONEHOT_FEATURES = [
    "is_action", "is_comedy", "is_drama", "is_horror",
    "is_sciencefiction", "is_animation", "is_romance", "is_thriller",
]

# primary_genre is the only string categorical we encode.
# director_name and lead_actor are represented via their performance-history
# numerics (director_avg_past_rating, lead_actor_avg_rating) which are already
# in NUMERIC_FEATURES – adding raw names would create hundreds of sparse
# dummies and risk overfitting on this small dataset.
CATEGORICAL_FEATURES = ["primary_genre"]

ALL_RAW_FEATURES = NUMERIC_FEATURES + GENRE_ONEHOT_FEATURES + CATEGORICAL_FEATURES


def encode_features(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str], StandardScaler, LabelEncoder]:
    """
    Encode categorical features and return processed DataFrames.

    Steps:
      1. LabelEncode 'primary_genre' (fit on train only).
      2. The genre one-hot flags and numeric features are already numeric.
      3. Return the processed feature matrices and the fitted encoders
         so callers can scale them separately per model.
    """
    le_genre = LabelEncoder()
    le_genre.fit(train_df["primary_genre"])

    def _encode(df):
        d = df[ALL_RAW_FEATURES].copy()
        d["primary_genre"] = le_genre.transform(df["primary_genre"])
        return d

    X_train = _encode(train_df)
    X_val   = _encode(val_df)
    X_test  = _encode(test_df)

    feature_cols = list(X_train.columns)
    return X_train, X_val, X_test, feature_cols, le_genre


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5: Temporal train / val / test split
# ─────────────────────────────────────────────────────────────────────────────
def temporal_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split df by release_year (no shuffling) to respect temporal ordering.

    Strategy:
      • Compute quantile year thresholds from the year distribution.
      • Train  = rows with year <= year_q70
      • Val    = rows with year_q70 < year <= year_q85
      • Test   = rows with year > year_q85

    This prevents data leakage: the model never sees "future" movies during
    training, matching real-world deployment where we predict before release.
    """
    years = df["release_year"].sort_values()
    y1 = int(np.quantile(years, TRAIN_QUANTILE))  # ~70th-percentile year
    y2 = int(np.quantile(years, VAL_QUANTILE))    # ~85th-percentile year

    train_df = df[df["release_year"] <= y1].copy()
    val_df   = df[(df["release_year"] > y1) & (df["release_year"] <= y2)].copy()
    test_df  = df[df["release_year"] > y2].copy()

    print(f"[temporal_split] Year thresholds: train ≤ {y1}, val {y1+1}–{y2}, test > {y2}")
    print(f"                 Sizes → train={len(train_df)}, val={len(val_df)}, test={len(test_df)}\n")
    return train_df, val_df, test_df


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6: Evaluation helpers
# ─────────────────────────────────────────────────────────────────────────────
def eval_classification(y_true, y_pred, label: str, class_names=None):
    """Print classification metrics: accuracy, macro F1, per-class report, confusion matrix."""
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average="macro")
    print(f"\n  [{label}] Accuracy={acc:.4f}  Macro-F1={f1:.4f}")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    print(f"  Confusion matrix (rows=true, cols=pred), classes={class_names}:")
    print(cm)
    return {"accuracy": acc, "macro_f1": f1}


def mape(y_true, y_pred, eps=1e-6):
    """Mean Absolute Percentage Error (safe against near-zero revenues)."""
    return np.mean(np.abs((np.array(y_true) - np.array(y_pred)) /
                           (np.abs(np.array(y_true)) + eps))) * 100


def eval_regression(y_true, y_pred, label: str, unit: str = ""):
    """Print regression metrics: RMSE, MAE, MAPE, R²."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape_val = mape(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"\n  [{label}] RMSE={rmse:,.2f}{unit}  MAE={mae:,.2f}{unit}  "
          f"MAPE={mape_val:.2f}%  R²={r2:.4f}")
    return {"rmse": rmse, "mae": mae, "mape": mape_val, "r2": r2}


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7: Baseline models (Algorithm 2 – scikit-learn linear)
# ─────────────────────────────────────────────────────────────────────────────
def train_baseline_models(
    X_train, X_val, X_test,
    y_verdict_train, y_verdict_val, y_verdict_test,
    y_revenue_train, y_revenue_val, y_revenue_test,
    y_rating_train,  y_rating_val,  y_rating_test,
    verdict_classes,
):
    """
    Train scikit-learn linear baselines for all three tasks.

    Scaling is critical for linear models; we fit StandardScaler on train only.
    XGBoost does NOT need scaling (tree-based), so we keep separate scalers.
    """
    print("\n" + "="*70)
    print("BASELINE MODELS (LogisticRegression / LinearRegression)")
    print("="*70)

    # ── Scale features (fit on train only) ───────────────────────────────
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(X_train)
    Xvl_s = scaler.transform(X_val)
    Xts_s = scaler.transform(X_test)

    # ── 7a. Verdict classification baseline ──────────────────────────────
    print("\n── 7a. Verdict Classification Baseline (LogisticRegression) ──")
    lr_clf = LogisticRegression(
        solver="lbfgs",
        max_iter=2000, C=1.0, random_state=SEED
    )
    lr_clf.fit(Xtr_s, y_verdict_train)

    print(" Validation:")
    eval_classification(y_verdict_val, lr_clf.predict(Xvl_s),
                        "LR Verdict – Val", verdict_classes)
    print(" Test:")
    eval_classification(y_verdict_test, lr_clf.predict(Xts_s),
                        "LR Verdict – Test", verdict_classes)

    # ── 7b. Revenue regression baseline ──────────────────────────────────
    print("\n── 7b. Revenue Regression Baseline (Ridge) ──")
    # Train Ridge on log-revenue for same footing as XGBoost, then back-transform.
    ridge_rev = Ridge(alpha=1.0)
    y_rev_log_train = np.log1p(y_revenue_train)
    ridge_rev.fit(Xtr_s, y_rev_log_train)

    rev_pred_val  = np.expm1(ridge_rev.predict(Xvl_s))
    rev_pred_test = np.expm1(ridge_rev.predict(Xts_s))

    print(" Validation:")
    eval_regression(y_revenue_val, rev_pred_val, "Ridge Revenue – Val", " USD")
    print(" Test:")
    eval_regression(y_revenue_test, rev_pred_test, "Ridge Revenue – Test", " USD")

    # ── 7c. Rating regression baseline ───────────────────────────────────
    print("\n── 7c. IMDb Rating Regression Baseline (Ridge) ──")
    ridge_rat = Ridge(alpha=1.0)
    ridge_rat.fit(Xtr_s, y_rating_train)

    print(" Validation:")
    eval_regression(y_rating_val, ridge_rat.predict(Xvl_s),
                    "Ridge Rating – Val")
    print(" Test:")
    eval_regression(y_rating_test, ridge_rat.predict(Xts_s),
                    "Ridge Rating – Test")

    return lr_clf, ridge_rev, ridge_rat, scaler


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8: XGBoost models (Algorithm 1)
# ─────────────────────────────────────────────────────────────────────────────
def _class_weights_dict(y_train: pd.Series) -> dict:
    """Compute inverse-frequency sample weights for imbalanced verdict classes."""
    counts = y_train.value_counts()
    n = len(y_train)
    return {cls: n / (len(counts) * cnt) for cls, cnt in counts.items()}


def train_xgboost_models(
    X_train, X_val, X_test,
    y_verdict_train, y_verdict_val, y_verdict_test,
    y_revenue_train, y_revenue_val, y_revenue_test,
    y_rating_train,  y_rating_val,  y_rating_test,
    verdict_classes,
    feature_cols,
):
    """
    Train XGBoost models for all three tasks with RandomizedSearchCV on
    the training set, selecting best params via validation-set evaluation.

    For verdict: encode labels to integers (XGBClassifier requirement).
    For revenue: optionally train on log-scale and exponentiate for eval.
    """
    print("\n" + "="*70)
    print("XGBOOST MODELS")
    print("="*70)

    # ── 8a. Verdict classification ────────────────────────────────────────
    print("\n── 8a. Verdict Classification (XGBClassifier) ──")

    # XGBClassifier requires integer labels
    le_verdict = LabelEncoder()
    le_verdict.fit(verdict_classes)          # ensures consistent ordering
    ytr_v = le_verdict.transform(y_verdict_train)
    yvl_v = le_verdict.transform(y_verdict_val)
    yts_v = le_verdict.transform(y_verdict_test)

    # Compute sample weights to handle class imbalance
    w_dict = _class_weights_dict(pd.Series(y_verdict_train))
    sample_w = np.array([w_dict[c] for c in y_verdict_train])

    param_dist_clf = {
        "n_estimators":  [100, 200, 300],
        "max_depth":     [3, 4, 5, 6],
        "learning_rate": [0.05, 0.1, 0.2],
        "subsample":     [0.7, 0.8, 1.0],
        "colsample_bytree": [0.7, 0.8, 1.0],
        "min_child_weight": [1, 3, 5],
    }

    base_clf = XGBClassifier(
        objective="multi:softmax",
        num_class=len(verdict_classes),
        eval_metric="mlogloss",
        use_label_encoder=False,
        random_state=SEED,
        verbosity=0,
    )

    # RandomizedSearchCV with 3-fold CV (still temporal within train set)
    rscv_clf = RandomizedSearchCV(
        base_clf, param_dist_clf,
        n_iter=20, cv=3, scoring="f1_macro",
        random_state=SEED, n_jobs=-1, refit=True
    )
    rscv_clf.fit(X_train, ytr_v, sample_weight=sample_w)
    best_clf_params = rscv_clf.best_params_
    print(f"  Best params (clf): {best_clf_params}")

    best_clf = rscv_clf.best_estimator_

    print(" Validation:")
    vl_pred_v = le_verdict.inverse_transform(best_clf.predict(X_val))
    eval_classification(y_verdict_val, vl_pred_v, "XGB Verdict – Val", verdict_classes)

    # Retrain on Train+Val combined for final test evaluation
    X_tv = pd.concat([X_train, X_val], ignore_index=True)
    y_tv = np.concatenate([ytr_v, yvl_v])
    sw_tv = np.concatenate([sample_w, np.array([w_dict[c] for c in y_verdict_val])])

    final_clf = XGBClassifier(**best_clf_params,
                               objective="multi:softmax",
                               num_class=len(verdict_classes),
                               eval_metric="mlogloss",
                               use_label_encoder=False,
                               random_state=SEED, verbosity=0)
    final_clf.fit(X_tv, y_tv, sample_weight=sw_tv)

    print(" Test (retrained on Train+Val):")
    ts_pred_v = le_verdict.inverse_transform(final_clf.predict(X_test))
    eval_classification(y_verdict_test, ts_pred_v, "XGB Verdict – Test", verdict_classes)

    # Save confusion matrix plot
    _plot_confusion_matrix(y_verdict_test, ts_pred_v, verdict_classes,
                           "XGBoost Verdict – Test Set",
                           os.path.join(OUTPUT_DIR, "cm_xgb_verdict_test.png"))

    # ── 8b. Revenue regression ─────────────────────────────────────────────
    print("\n── 8b. Revenue Regression (XGBRegressor) ──")

    # Train on log-revenue for better numerical stability;
    # predictions are exponentiated back to USD for evaluation.
    if TRAIN_REVENUE_LOG:
        ytr_r = np.log1p(y_revenue_train)
        yvl_r = np.log1p(y_revenue_val)
    else:
        ytr_r = y_revenue_train
        yvl_r = y_revenue_val

    param_dist_reg = {
        "n_estimators":  [100, 200, 300],
        "max_depth":     [3, 4, 5, 6],
        "learning_rate": [0.05, 0.1, 0.2],
        "subsample":     [0.7, 0.8, 1.0],
        "colsample_bytree": [0.7, 0.8, 1.0],
        "min_child_weight": [1, 3, 5],
    }

    base_reg_rev = XGBRegressor(
        objective="reg:squarederror",
        random_state=SEED, verbosity=0
    )
    rscv_rev = RandomizedSearchCV(
        base_reg_rev, param_dist_reg,
        n_iter=20, cv=3, scoring="neg_root_mean_squared_error",
        random_state=SEED, n_jobs=-1, refit=True
    )
    rscv_rev.fit(X_train, ytr_r)
    best_rev_params = rscv_rev.best_params_
    print(f"  Best params (revenue): {best_rev_params}")

    best_reg_rev = rscv_rev.best_estimator_

    def _rev_predict(model, X):
        p = model.predict(X)
        return np.expm1(p) if TRAIN_REVENUE_LOG else p

    print(" Validation:")
    eval_regression(y_revenue_val, _rev_predict(best_reg_rev, X_val),
                    "XGB Revenue – Val", " USD")

    # Retrain on Train+Val
    X_tv_np = pd.concat([X_train, X_val], ignore_index=True)
    y_tv_r  = np.concatenate([ytr_r, yvl_r])
    final_reg_rev = XGBRegressor(**best_rev_params,
                                  objective="reg:squarederror",
                                  random_state=SEED, verbosity=0)
    final_reg_rev.fit(X_tv_np, y_tv_r)

    print(" Test (retrained on Train+Val):")
    eval_regression(y_revenue_test, _rev_predict(final_reg_rev, X_test),
                    "XGB Revenue – Test", " USD")

    # Feature importance plot
    _plot_feature_importance(final_reg_rev, feature_cols,
                             "XGBoost Revenue – Feature Importance (Gain)",
                             os.path.join(OUTPUT_DIR, "fi_xgb_revenue.png"))

    # ── 8c. IMDb rating regression ─────────────────────────────────────────
    print("\n── 8c. IMDb Rating Regression (XGBRegressor) ──")

    base_reg_rat = XGBRegressor(
        objective="reg:squarederror",
        random_state=SEED, verbosity=0
    )
    rscv_rat = RandomizedSearchCV(
        base_reg_rat, param_dist_reg,
        n_iter=20, cv=3, scoring="neg_root_mean_squared_error",
        random_state=SEED, n_jobs=-1, refit=True
    )
    rscv_rat.fit(X_train, y_rating_train)
    best_rat_params = rscv_rat.best_params_
    print(f"  Best params (rating): {best_rat_params}")

    best_reg_rat = rscv_rat.best_estimator_

    print(" Validation:")
    eval_regression(y_rating_val, best_reg_rat.predict(X_val),
                    "XGB Rating – Val")

    # Retrain on Train+Val
    final_reg_rat = XGBRegressor(**best_rat_params,
                                  objective="reg:squarederror",
                                  random_state=SEED, verbosity=0)
    final_reg_rat.fit(X_tv_np, np.concatenate([y_rating_train, y_rating_val]))

    print(" Test (retrained on Train+Val):")
    eval_regression(y_rating_test, final_reg_rat.predict(X_test),
                    "XGB Rating – Test")

    # Feature importance plot
    _plot_feature_importance(final_reg_rat, feature_cols,
                             "XGBoost Rating – Feature Importance (Gain)",
                             os.path.join(OUTPUT_DIR, "fi_xgb_rating.png"))

    return final_clf, final_reg_rev, final_reg_rat, le_verdict


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9: SHAP explainability (optional)
# ─────────────────────────────────────────────────────────────────────────────
def run_shap_analysis(model, X_sample: pd.DataFrame, feature_cols: list, tag: str):
    """
    Compute and plot SHAP values for an XGBoost model.
    Uses TreeExplainer (fast, exact for tree models).
    Pass a small sample (e.g., 100 rows) to keep runtime manageable.
    """
    if not USE_SHAP:
        return

    print(f"\n[SHAP] Computing SHAP values for {tag} on {len(X_sample)} samples …")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

    # For multi-output (classification) shap_values is a list; take class 0 for simplicity
    if isinstance(shap_values, list):
        sv = shap_values[0]
    else:
        sv = shap_values

    plt.figure()
    shap.summary_plot(sv, X_sample, feature_names=feature_cols, show=False)
    plt.title(f"SHAP Summary – {tag}")
    plt.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, f"shap_summary_{tag.replace(' ','_')}.png")
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"[SHAP] Saved summary plot → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10: Plotting helpers
# ─────────────────────────────────────────────────────────────────────────────
def _plot_confusion_matrix(y_true, y_pred, classes, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"  [plot] Confusion matrix saved → {save_path}")


def _plot_feature_importance(model, feature_cols, title, save_path):
    scores = model.get_booster().get_fscore()   # gain-based importance
    imp_df = pd.DataFrame(
        [(f, scores.get(f, 0)) for f in feature_cols],
        columns=["feature", "importance"]
    ).sort_values("importance", ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.barplot(data=imp_df, x="importance", y="feature", ax=ax, palette="viridis")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"  [plot] Feature importance saved → {save_path}")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 11: Save models
# ─────────────────────────────────────────────────────────────────────────────
def save_models(models: dict, output_dir: str):
    """Persist models to disk using joblib."""
    for name, obj in models.items():
        path = os.path.join(output_dir, f"{name}.pkl")
        joblib.dump(obj, path)
        print(f"  [save] {name} → {path}")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 12: Main orchestration
# ─────────────────────────────────────────────────────────────────────────────
def main():
    # ── 12a. Load & inspect ───────────────────────────────────────────────
    df = load_data(CSV_PATH)

    # ── 12b. Construct targets ────────────────────────────────────────────
    df = add_targets(df)

    # ── 12c. Temporal split ───────────────────────────────────────────────
    train_df, val_df, test_df = temporal_split(df)

    # ── 12d. Encode features (fit encoders on train only) ─────────────────
    X_train, X_val, X_test, feature_cols, le_genre = encode_features(
        train_df, val_df, test_df
    )
    print(f"[encode] Feature columns ({len(feature_cols)}): {feature_cols}\n")

    # ── 12e. Extract target arrays ─────────────────────────────────────────
    VERDICT_CLASSES = ["Flop", "Average", "Hit"]  # consistent label order

    y_verdict_train = train_df["verdict_label"].values
    y_verdict_val   = val_df["verdict_label"].values
    y_verdict_test  = test_df["verdict_label"].values

    y_revenue_train = train_df["revenue_usd"].values
    y_revenue_val   = val_df["revenue_usd"].values
    y_revenue_test  = test_df["revenue_usd"].values

    y_rating_train  = train_df["imdb_rating"].values
    y_rating_val    = val_df["imdb_rating"].values
    y_rating_test   = test_df["imdb_rating"].values

    # ── 12f. Baseline models ──────────────────────────────────────────────
    lr_clf, ridge_rev, ridge_rat, scaler = train_baseline_models(
        X_train, X_val, X_test,
        y_verdict_train, y_verdict_val, y_verdict_test,
        y_revenue_train, y_revenue_val, y_revenue_test,
        y_rating_train,  y_rating_val,  y_rating_test,
        VERDICT_CLASSES,
    )

    # ── 12g. XGBoost models ───────────────────────────────────────────────
    xgb_clf, xgb_rev, xgb_rat, le_verdict = train_xgboost_models(
        X_train, X_val, X_test,
        y_verdict_train, y_verdict_val, y_verdict_test,
        y_revenue_train, y_revenue_val, y_revenue_test,
        y_rating_train,  y_rating_val,  y_rating_test,
        VERDICT_CLASSES,
        feature_cols,
    )

    # ── 12h. SHAP analysis on test sample ─────────────────────────────────
    # Use at most 100 test rows to keep SHAP fast
    shap_sample = X_test.iloc[:min(100, len(X_test))].reset_index(drop=True)
    run_shap_analysis(xgb_rev, shap_sample, feature_cols, "Revenue XGBoost")
    run_shap_analysis(xgb_rat, shap_sample, feature_cols, "Rating XGBoost")

    # ── 12i. Save models ──────────────────────────────────────────────────
    print("\n[Saving models]")
    save_models({
        "baseline_lr_verdict": lr_clf,
        "baseline_ridge_revenue": ridge_rev,
        "baseline_ridge_rating": ridge_rat,
        "baseline_scaler": scaler,
        "xgb_verdict_classifier": xgb_clf,
        "xgb_revenue_regressor": xgb_rev,
        "xgb_rating_regressor": xgb_rat,
        "label_encoder_verdict": le_verdict,
        "label_encoder_genre": le_genre,
    }, OUTPUT_DIR)

    print("\n✅  CinePredict pipeline complete. Outputs saved to:", OUTPUT_DIR)


if __name__ == "__main__":
    main()

[load_data] Loaded 292 rows × 24 columns.
            Year range: 1972 – 2016
            Columns: ['movie_id', 'title', 'budget', 'budget_log', 'revenue_usd', 'release_year', 'release_month', 'is_action', 'is_comedy', 'is_drama', 'is_horror', 'is_sciencefiction', 'is_animation', 'is_romance', 'is_thriller', 'num_genres', 'primary_genre', 'director_name', 'lead_actor', 'director_avg_past_rating', 'lead_actor_avg_rating', 'sidecast_avg_rating', 'genre_trend_score', 'imdb_rating']

[add_targets] Verdict label distribution:
verdict_label
Hit        146
Flop        82
Average     64
Name: count, dtype: int64 

[temporal_split] Year thresholds: train ≤ 2008, val 2009–2011, test > 2011
                 Sizes → train=212, val=39, test=41

[encode] Feature columns (18): ['budget', 'budget_log', 'release_year', 'release_month', 'num_genres', 'director_avg_past_rating', 'lead_actor_avg_rating', 'sidecast_avg_rating', 'genre_trend_score', 'is_action', 'is_comedy', 'is_drama', 'is_horror', 'is_sci